# 1. Code Chunking 
Strategy: Use AST (Abstract Syntax Tree) parsing. 

Rule: Never split a function mid-body. Keep imports and class declarations with their methods.

In [9]:
import ast

def chunk_python(code):

    tree = ast.parse(code)

    chunks = []

    for node in tree.body:

        chunk = ast.get_source_segment(
            code,
            node
        )

        chunks.append(chunk)

    return chunks

In [18]:
text = '''
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode


# Step 1: Generate an AIMessage that may include a tool-call to be sent.
def query_or_respond(state: MessagesState):
    \"\"\"Generate tool call for retrieval or respond.\"\"\"
    llm_with_tools = llm.bind_tools([retrieve])
    response = llm_with_tools.invoke(state["messages"])
    # MessagesState appends messages to state instead of overwriting
    return {"messages": [response]}


# Step 2: Execute the retrieval.
tools = ToolNode([retrieve])


# Step 3: Generate a response using the retrieved content.
def generate(state: MessagesState):
    \"\"\"Generate answer.\"\"\"
    # Get generated ToolMessages
    recent_tool_messages = []
    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]

    # Format into prompt
    docs_content = " ".join(doc.content for doc in tool_messages)
    system_message_content = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer "
        "the question. If you don't know the answer, say that you "
        "don't know. Use three sentences maximum and keep the "
        "answer concise."
        f"{docs_content}"
    )
    conversation_messages = [
        message
        for message in state["messages"]
        if message.type in ("human", "system")
        or (message.type == "ai" and not message.tool_calls)
    ]
    prompt = [SystemMessage(system_message_content)] + conversation_messages

    # Run
    response = llm.invoke(prompt)
    return {"messages": [response]}
'''

In [19]:
chunk_python(text)

['from langchain_core.messages import SystemMessage',
 'from langgraph.prebuilt import ToolNode',
 'def query_or_respond(state: MessagesState):\n    """Generate tool call for retrieval or respond."""\n    llm_with_tools = llm.bind_tools([retrieve])\n    response = llm_with_tools.invoke(state["messages"])\n    # MessagesState appends messages to state instead of overwriting\n    return {"messages": [response]}',
 'tools = ToolNode([retrieve])',
 'def generate(state: MessagesState):\n    """Generate answer."""\n    # Get generated ToolMessages\n    recent_tool_messages = []\n    for message in reversed(state["messages"]):\n        if message.type == "tool":\n            recent_tool_messages.append(message)\n        else:\n            break\n    tool_messages = recent_tool_messages[::-1]\n\n    # Format into prompt\n    docs_content = " ".join(doc.content for doc in tool_messages)\n    system_message_content = (\n        "You are an assistant for question-answering tasks. "\n        "Use 

# Better: Keep Imports with Functions

In [20]:
import ast

def chunk_python(code):

    tree = ast.parse(code)

    imports = []

    chunks = []

    for node in tree.body:

        if isinstance(
            node,
            (ast.Import, ast.ImportFrom)
        ):
            imports.append(
                ast.get_source_segment(code, node)
            )

    import_block = "\n".join(imports)

    for node in tree.body:

        if isinstance(
            node,
            (ast.FunctionDef,
             ast.AsyncFunctionDef)
        ):

            func_code = ast.get_source_segment(
                code,
                node
            )

            chunks.append(
                import_block +
                "\n\n" +
                func_code
            )

    return chunks

In [21]:
chunk_python(text)

['from langchain_core.messages import SystemMessage\nfrom langgraph.prebuilt import ToolNode\n\ndef query_or_respond(state: MessagesState):\n    """Generate tool call for retrieval or respond."""\n    llm_with_tools = llm.bind_tools([retrieve])\n    response = llm_with_tools.invoke(state["messages"])\n    # MessagesState appends messages to state instead of overwriting\n    return {"messages": [response]}',
 'from langchain_core.messages import SystemMessage\nfrom langgraph.prebuilt import ToolNode\n\ndef generate(state: MessagesState):\n    """Generate answer."""\n    # Get generated ToolMessages\n    recent_tool_messages = []\n    for message in reversed(state["messages"]):\n        if message.type == "tool":\n            recent_tool_messages.append(message)\n        else:\n            break\n    tool_messages = recent_tool_messages[::-1]\n\n    # Format into prompt\n    docs_content = " ".join(doc.content for doc in tool_messages)\n    system_message_content = (\n        "You are an

## Tree-Sitter (Industry Standard)

Most production code RAG systems don't use Python's ast.

They use Tree-sitter.

Why?

Python AST:
Python only

Tree-sitter:
Python
Java
Go
Rust
JS
TS
C++
Java
etc.